#### Cleaning Section: [cleaning_report.md](../docs/Cleaning_logs/cleaning_report.md)

- `Early Data Analysis`
- `Duplicate Correction`
- `Inconsistencies Fixing`
- `Missing Data Checking`
- `Planning for EDA`

In [1]:
# Data was given in snippets, we have to concat everything into one df
import pandas as pd
import plotly.express as px

years = list(range(2020, 2027))
dfs = []

for year in years:
    for quarter in ["Q1", "Q2", "Q3", "Q4"]:
        file_path = f"../data/raw/BDD PRODUCCION/BDD PRODUCCION/{year}/{quarter} {year}.csv"
        try:
            df = pd.read_csv(file_path)

            if year == 2025 and quarter == "Q2":
                s = df["order_date"]
                p1 = pd.to_datetime(s, dayfirst=True, errors="coerce")
                p2 = pd.to_datetime(s, dayfirst=False, errors="coerce")
                num = pd.to_numeric(s, errors="coerce")
                excel = pd.to_datetime(num, unit="D", origin="1899-12-30")
                # Fill order_date with p2 first, then p1, and finally excel if both are NaT 
                df["order_date"] = p2.fillna(p1).fillna(excel)
            else:
                df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

            dfs.append(df)
        except FileNotFoundError as e:
            print(f"File not found: {file_path}, error: {e}")

remissions = pd.concat(dfs, ignore_index=True)
remissions.head()


File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv'


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04,1028,2020-02-04 08:00:00,9631,510,2.5,2020-02-04 07:37:56,2020-02-04 08:55:49,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04,1026,2020-02-04 08:30:00,10145,510,3.5,2020-02-04 08:09:01,2020-02-04 09:09:15,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5


DO NOT TOUCH THIS CODE (Fixes issue with format of `Q2 2025.csv`)

In [2]:
order_date = pd.to_datetime(remissions["order_date"], errors="coerce").dt.normalize()

def rebuild_datetime(col):
    raw = remissions[col].astype(str).str.strip()

    time_str = raw.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?\s*[APMapm]{0,2})")[0]
    time_str = time_str.str.replace(r"\.\d+", "", regex=True)

    t24 = pd.to_datetime(time_str, format="%H:%M:%S", errors="coerce")
    t24 = t24.fillna(pd.to_datetime(time_str, format="%H:%M", errors="coerce"))

    t12 = pd.to_datetime(time_str, format="%I:%M:%S %p", errors="coerce")
    t12 = t12.fillna(pd.to_datetime(time_str, format="%I:%M %p", errors="coerce"))

    t = t24.fillna(t12)
    time_only = t - t.dt.normalize()

    return order_date + time_only

remissions["typed_time"] = rebuild_datetime("typed_time")
remissions["start_time"] = rebuild_datetime("start_time")
remissions["at_plant_time"] = rebuild_datetime("at_plant_time")

In [3]:
# show rows from May 2025 
remissions_may = remissions[(remissions["order_date"].dt.month == 5) & (remissions["order_date"].dt.year == 2025)]
remissions_may.head()

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
295158,2025,51008733,2025-05-02,1212,2025-05-02 07:00:00,7043,510,3.5,2025-05-02 06:15:15,2025-05-02 08:05:51,110,ONE TIME OCTAVIO RIOS,NORMA CHAPARRO VAZQUEZ,CALLE BOSQUE DE PIEDRA #2828 BOSQUE DE,CHG-2
295159,2025,51008735,2025-05-02,1223,2025-05-02 07:00:00,9431,510,3.5,2025-05-02 06:15:52,2025-05-02 07:55:00,100,CASA HOGAR PARA NIÑOS YIREH,CASA HOGAR PARA NIÑOS YIREH,BAHIA DE SAN QUINTIN 9513 COL RINCONADAS,CHK-3
295160,2025,51008737,2025-05-02,1227,2025-05-02 07:15:00,6605,510,3.0,2025-05-02 06:27:16,2025-05-02 08:33:00,126,LEONEL RAMIREZ JARAMILLO,LEONEL RAMIREZ JARAMILLO,CIRCUITO PODERES #10702 SAN GABRIEL ETA,CHD-4
295161,2025,51008739,2025-05-02,1423,2025-05-02 07:45:00,12193,510,6.5,2025-05-02 07:47:03,2025-05-02 09:17:00,90,JONATHAN MARQUEZ BACA,OBRAS VARIAS,CALLE ENRIQUE MULLER & LEONA VICARIO RE,CHK-5
295162,2025,51008740,2025-05-02,1214,2025-05-02 08:15:00,9431,510,2.0,2025-05-02 07:55:56,2025-05-02 08:37:00,42,INDUSTRIAS CALCITE,AV TECNOLOGICO,AV TECNOLOGICO 11708 REVOLUCION CHIHU,CHJ-6


#### Checking issue with some data corruption from plant 710 

In [13]:
# Diagnose how Q2 2025 dates are parsed around May 2-3 vs Apr 6-7 for p1 (raw q2) and p2 (remmissions)
remissions_window = remissions.loc[
    (remissions["order_date"] >= "2025-04-01")
    & (remissions["order_date"] < "2025-05-10"),
    "order_date"
].dt.date.value_counts().sort_index()
remissions_window.head(40)

raw_q2_2025 = pd.read_csv("../data/raw/BDD PRODUCCION/BDD PRODUCCION/2025/Q2 2025.csv")
raw_q2_2025["order_date_raw"] = raw_q2_2025["order_date"].astype(str).str.strip()

raw_q2_2025["p1"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=True, errors="coerce")
raw_q2_2025["p2"] = pd.to_datetime(raw_q2_2025["order_date_raw"], dayfirst=False, errors="coerce")
raw_q2_2025["num"] = pd.to_numeric(raw_q2_2025["order_date_raw"], errors="coerce")
raw_q2_2025["excel"] = pd.to_datetime(raw_q2_2025["num"], unit="D", origin="1899-12-30")

def sample_range(series, start, end):
    mask = (series >= start) & (series < end)
    cols = ["order_date_raw", "p1", "p2", "excel"]
    return raw_q2_2025.loc[mask, cols].head(20)

print("p1 May 2-3:")
display(sample_range(raw_q2_2025["p1"], "2025-05-02", "2025-05-04"))

print("p2 May 2-3:")
display(sample_range(raw_q2_2025["p2"], "2025-05-02", "2025-05-04"))

print("excel May 2-3:")
display(sample_range(raw_q2_2025["excel"], "2025-05-02", "2025-05-04"))

print("p1 Apr 6-7:")
display(sample_range(raw_q2_2025["p1"], "2025-04-06", "2025-04-08"))

print("p2 Apr 6-7:")
display(sample_range(raw_q2_2025["p2"], "2025-04-06", "2025-04-08"))

print("excel Apr 6-7:")
display(sample_range(raw_q2_2025["excel"], "2025-04-06", "2025-04-08"))

p1 May 2-3:


,order_date_raw,p1,p2,excel


p2 May 2-3:


,order_date_raw,p1,p2,excel
0,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
1,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
2,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
3,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
4,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
5,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
6,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
7,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
8,5/2/2025 0:00,2025-02-05,2025-05-02,NaT
9,5/2/2025 0:00,2025-02-05,2025-05-02,NaT


excel May 2-3:


...

,order_date_raw,p1,p2,excel
1091,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1092,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1093,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1094,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1095,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1096,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1097,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1098,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1099,6/4/2025 0:00,2025-04-06,2025-06-04,NaT
1100,6/4/2025 0:00,2025-04-06,2025-06-04,NaT


p2 Apr 6-7:


,order_date_raw,p1,p2,excel


excel Apr 6-7:


,order_date_raw,p1,p2,excel


1. Analysis 

In [4]:
# Check data types
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  int64         
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  int64         
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  int64         
 6   ship_plant_code      356332 non-null  int64         
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [5]:
# Change 'order_date' and 'typed_time' to datetime format
remissions['order_date'] = pd.to_datetime(remissions['order_date'], errors='raise', format='mixed')
remissions['typed_time'] = pd.to_datetime(remissions['typed_time'], errors='raise', format='mixed')
remissions['start_time'] = pd.to_datetime(remissions['start_time'], errors='raise', format='mixed')
remissions['at_plant_time'] = pd.to_datetime(remissions['at_plant_time'], errors='raise', format='mixed')
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  int64         
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  int64         
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  int64         
 6   ship_plant_code      356332 non-null  int64         
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [7]:
# Change `ship_plant_code`, `tkt_code`, `order_code` and `truck_code` to string type
remissions['ship_plant_code'] = remissions['ship_plant_code'].astype(str)
remissions['tkt_code'] = remissions['tkt_code'].astype(str)
remissions['order_code'] = remissions['order_code'].astype(str)
remissions['truck_code'] = remissions['truck_code'].astype(str)
remissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 356332 entries, 0 to 356331
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 356332 non-null  int64         
 1   tkt_code             356332 non-null  str           
 2   order_date           356332 non-null  datetime64[us]
 3   order_code           356332 non-null  str           
 4   start_time           356332 non-null  datetime64[us]
 5   truck_code           356332 non-null  str           
 6   ship_plant_code      356332 non-null  str           
 7   u_Volumen            356332 non-null  float64       
 8   typed_time           356332 non-null  datetime64[us]
 9   at_plant_time        356332 non-null  datetime64[us]
 10  u_Cicle              356332 non-null  int64         
 11  name                 356331 non-null  str           
 12  Nombre del proyecto  356332 non-null  str           
 13  ship_addr_line       3562

In [8]:
# We don't need column `Year`
remissions = remissions.drop(columns=["Year"]) # One-time use
remissions.head(1)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1


Null/Missing data check

In [9]:
# Check for null values in each column
nulls = remissions.isnull().sum()
nulls = nulls[nulls > 0]  # keep only columns with at least 1 null

null_summary = pd.DataFrame({
    "null_count": nulls,
    "null_%": (nulls / len(remissions) * 100).round(2)
}).sort_values("null_count", ascending=False)

null_summary

,null_count,null_%
map_page,444,0.12
ship_addr_line,85,0.02
name,1,0.00


In [10]:
# It´s okay to drop rows with null values (<2%)
remissions = remissions[remissions['ship_addr_line'].notnull() & remissions['map_page'].notnull() & remissions['name'].notnull()]
print("Number of rows after dropping null values:", remissions.shape[0])

Number of rows after dropping null values: 355802


In [11]:
# Check missing data over time
# Analyze hourly distribution of u_Volumen by ship_plant_code
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)["u_Volumen"]
    .sum()
)

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="u_Volumen",
    color="ship_plant_code",
    title="Hourly Distribution of u_Volumen by Ship Plant Code (2020-2026)"
)
fig.show()

In [12]:
hourly_distribution = (
    remissions
    .assign(hour_bucket=remissions["typed_time"].dt.floor("h"))
    .groupby(["hour_bucket", "ship_plant_code"], as_index=False)
    .size()
    .rename(columns={"size": "remission_count"})
)

hourly_distribution["u_Volumen"] = hourly_distribution["remission_count"]

fig = px.line(
    hourly_distribution,
    x="hour_bucket",
    y="remission_count",
    color="ship_plant_code",
    title="Hourly Distribution of Remissions by Ship Plant Code (2020-2026)"
)
fig.show()

#### General Missing Data Observations:
- The common no-data-registered periods for all plants are:
    + `January 1st 2022` - `Feb 2nd 2022`
        + **Most likely reason**: Winter Vacations
    + `December 31 2023` - `Feb 1st 2024`
        + **Most likely reason**: Winter Vacations
    + `Mar 31 2025` - `May 2nd 2025`
- There are NO apparent stop from working during `2022-2023` winter vacations period nor `2024-2025`

- **Action taken:** After finishing cleaning phase I will apply imputation methods based on historic data and similar plants. [Data_Imputation](./2.data_imputation.ipynb)

#### Missing Data per plant:
- Plant 514
    + Begins to register since `July 20 2023 
    --- **Action Taken** ---
    + No action, TFT is not affected for this kind of "late" data

- Plant 710
    + Massive gap between `May 2023` - `Feb 2024`
    + Probably one outlier during `June 2021`
    --- **Action Taken** ---
    + After cleaning, I will apply imputation methods based on historic data and similar plants. [Data_Imputation](./2.data_imputation.ipynb)

- Plant 512
    + Heavy downfall in remissions and volume between `April 2020` and `June 15 2020` most likely due to pandemic
    --- **Action Taken** ---
    + Because it is the only 


#### Other Observations
- `April 2025` has no data in the given raw dataset, but there are rows for `April 6` and `April 7` of said year, also data from `May 2` and `May 3` is not appearing.
- Outliers to check:
    + Too many orders and volume in `Sept 5 2024 10:00am`
    + Some outliers for late 2025

In [86]:
# Duplicate check for all time columns
typed_time_duplicates = remissions[remissions.duplicated(subset=['start_time', 'at_plant_time','truck_code'], keep=False)]
typed_time_duplicates = typed_time_duplicates.sort_values(by='typed_time', ascending=True)
typed_time_duplicates.head(10)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [87]:
print("Number of duplicate rows based on start_time, at_plant_time, and truck_code:", typed_time_duplicates.shape[0])

Number of duplicate rows based on start_time, at_plant_time, and truck_code: 0


In [88]:
# It is impossible for many trucks to have the same start_time and at_plant_time
# Therefore, these are duplicates that should be removed, as they are likely to be errors in the data entry process.
remissions = remissions.drop(typed_time_duplicates.index)
print("Number of rows after dropping duplicates:", remissions.shape[0])

Number of rows after dropping duplicates: 355802


We will check u_Cicle to see if there is any order that was never completed.

In [89]:
# Graph the distribution of u_Cicle to see if there are any outliers
fig = px.histogram(remissions, x='u_Cicle', nbins=len(remissions['u_Cicle'].unique()), title='Distribution of u_Cicle')
fig.show()

In [90]:
# Boxplot of u_Cicle to check for outliers
fig = px.box(remissions, y='u_Cicle', title='Boxplot of u_Cicle')
fig.show()

We will check distribution of volume amount

In [91]:
# Calculate counts and percentages for u_Volumen
counts = remissions['u_Volumen'].value_counts().sort_index()
percentages = (counts / counts.sum()) * 100

# Histogram of frequency of each unique volume value to check for outliers
x_labels = counts.index.astype(str)  # treat volumes as categorical so bars are wider
fig = px.bar(x=x_labels, y=counts.values, title='Distribution of Volume',
             labels={'x':'u_Volumen','y':'count'}, text=counts.values)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    customdata=percentages.values,
    hovertemplate='<b>Volume:</b> %{x}<br><b>Count:</b> %{y}<br><b>Percentage:</b> %{customdata:.2f}%<extra></extra>'
)
fig.update_layout(
    bargap=0.1,        # reduce gap between bars
    xaxis_tickangle=45,
    width=1000
)
fig.show()

In [92]:
# Sum all the tkt_code count and u_Volumen related to each order_code and print the orders with highest to lowest total volume and remissions
order_volume = remissions.groupby('order_code').agg(
    u_Volumen=('u_Volumen', 'sum'),
    tkt_code_sum=('tkt_code', 'count')
).reset_index()

order_volume = order_volume.sort_values(by='u_Volumen', ascending=False)
print(order_volume.head(10))

    order_code  u_Volumen  tkt_code_sum
0         1000    5448.00          1148
113       1113    4790.00          1313
86        1086    4750.50          1232
70        1070    4717.00          1203
107       1107    4482.00          1203
1         1001    4441.50           970
54        1054    4386.25          1113
7         1007    4300.50          1018
128       1128    4228.00          1195
49        1049    4153.50          1081


In [93]:
# print the count of different map_page values associated with the order_code = 1000
order_1000_map_pages = remissions[remissions['order_code'] == 1000]['map_page'].nunique()
print("Count of different map_page values associated with order_code 1000:", order_1000_map_pages)

Count of different map_page values associated with order_code 1000: 0


In [94]:
# print the oldest date of the tkt_code and the newest of the order_code = 1000
order_1000_dates = remissions[remissions['order_code'] == 1000]['order_date']
oldest_date = order_1000_dates.min()
newest_date = order_1000_dates.max()
print("Oldest date of tkt_code for order_code 1000:", oldest_date)
print("Newest date of tkt_code for order_code 1000:", newest_date)

Oldest date of tkt_code for order_code 1000: NaT
Newest date of tkt_code for order_code 1000: NaT


In [95]:
# print the count of how many unique order_code are in total
unique_order_codes = remissions['order_code'].nunique()
print("Count of unique order_code in total:", unique_order_codes)

Count of unique order_code in total: 795


In [96]:
# Distribution with x values being the order_code unique values and y values being the count of tkt_code for each order_code
order_code_counts = remissions['order_code'].value_counts().sort_index()

fig = px.bar(
    x=order_code_counts.index.astype(str),
    y=order_code_counts.values,
    title='Distribution of tkt_code count by order_code',
    labels={'x': 'order_code', 'y': 'tkt_code count'},
    text=order_code_counts.values
)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    hovertemplate='<b>Order Code:</b> %{x}<br><b>tkt_code Count:</b> %{y}<extra></extra>'
)
fig.update_layout(
    bargap=0.1,
    xaxis_showticklabels=False,
    width=1000
)
fig.show()


In [97]:
# make a boxplot of the previous graph to check for outliers
fig = px.box(order_code_counts.values, title='Boxplot of tkt_code count by order_code')
fig.update_layout(width=800)
fig.show()

In [98]:
# Check distribution of plants where u_Cicle is 1
u_cicle_1 = remissions[remissions['u_Cicle'] == 1]
plant_counts = u_cicle_1['ship_plant_code'].value_counts()
print(plant_counts)

ship_plant_code
512    4220
515    1715
514    1458
710    1372
511     783
510     324
717     203
Name: count, dtype: int64


In [99]:
#Looks like the tkt_codes can be repeated after years of orders.
remissions[remissions['tkt_code'].duplicated(keep=False)].sort_values('tkt_code', ascending=True).head(6)


,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
1,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
310446,51013233,2025-08-02,1035,2025-08-02 07:00:00,9430,510,2.0,2025-08-02 06:46:30,2025-08-02 07:46:17,60,ONE TIME PROMOCIONES OCTAVIO RIOS,MARGARITO ROMERO,ING. CARRILLO 16344 COL TRAHUMARA,CHJ-3
3,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
310448,51013236,2025-08-02,1201,2025-08-02 07:00:00,7043,510,4.0,2025-08-02 06:54:06,2025-08-02 08:14:23,80,JONATHAN MARQUEZ BACA,OBRAS VARIAS,ARROYO NARAGUA #2225 LOS ARROYOS,CHD-5
5,51013239,2020-02-04,1150,2020-02-04 09:00:00,6611,510,6.0,2020-02-04 08:36:24,2020-02-04 09:30:24,54,JULIO ARMANDO HINOJOS ENRIQUEZ,FRAC CALZADA DEL BOSQUE AGH,FRAC CALZADA DEL BOSQUE AGH FRAC CAL,CH-H3
310450,51013239,2025-08-02,1090,2025-08-02 08:00:00,9430,510,6.5,2025-08-02 07:46:56,2025-08-02 09:03:00,77,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,MONTE HIMALAYA 4341 QUINTAS CAROLINA,CHH-8


We will now check the consistency of remissions per plant

In [100]:
print(remissions['ship_plant_code'].unique())
# The new plant (717) was not used for the past version of this project

<ArrowStringArray>
['510', '511', '512', '515', '710', '717', '514']
Length: 7, dtype: str


In [101]:
counts = remissions['ship_plant_code'].value_counts()
percentages = (counts / counts.sum()) * 100
print(counts, '\t', percentages.round(2)) # percentages
print("total:", counts.sum())

ship_plant_code
512    82691
510    71417
511    61025
515    52454
710    45916
514    40294
717     2005
Name: count, dtype: int64 	 ship_plant_code
512    23.24
510    20.07
511    17.15
515    14.74
710    12.90
514    11.32
717     0.56
Name: count, dtype: float64
total: 355802


In [102]:
# See lowest and highest datetime for each plant
for plant_code in remissions['ship_plant_code'].unique():
    plant = remissions[remissions['ship_plant_code'] == plant_code]
    print(f"Plant {plant_code}:")
    print(f"  Lowest datetime: {plant['start_time'].min()}")
    print(f"  Highest datetime: {plant['start_time'].max()}")

Plant 510:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 09:30:00
Plant 511:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:00:00
Plant 512:
  Lowest datetime: 2020-02-04 05:00:00
  Highest datetime: 2026-04-24 11:30:00
Plant 515:
  Lowest datetime: 2020-02-04 08:30:00
  Highest datetime: 2026-04-24 10:00:00
Plant 710:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-04-24 10:15:00
Plant 717:
  Lowest datetime: 2020-02-04 07:30:00
  Highest datetime: 2026-03-20 08:00:00
Plant 514:
  Lowest datetime: 2022-07-20 07:00:00
  Highest datetime: 2026-04-24 10:45:00


In [103]:
# Drop plant 717 due to low number of remissions and being inactive for the past 4 years
remissions_df = remissions[remissions['ship_plant_code'] != 717]
print("Number of rows after dropping plant 717:", remissions.shape[0])

# Also plant 514 starts from 2022, but that is no problem as it covers over a 10% of the total remission count

Number of rows after dropping plant 717: 355802


In [104]:
# There are volume outliers based on EDA, so we will find them and drop them
# print the rows in which `u_Volumen` is greater than 7 or less/equal than 0
outliers_volumen = remissions[(remissions['u_Volumen'] > 7) | (remissions['u_Volumen'] <= 0)]
print("Rows with volume outliers:")
print(outliers_volumen.shape[0])

Rows with volume outliers:
0


In order to identify repeated orders, the row would need to have repeated values in the following columns: order_code, order_date, typed_time, truck_code. 

In [105]:
# Look for repeated order_codes

repeated_orders = remissions[remissions.duplicated(subset=['order_code', 'order_date','typed_time','truck_code'], keep=False)]
repeated_orders.head(-1)

# No repeated orders.

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [106]:
# Count unique values for each column
unique_counts = remissions.nunique()
print(unique_counts)

tkt_code               335876
order_date               1755
order_code                795
start_time              70500
truck_code                151
ship_plant_code             7
u_Volumen                  34
typed_time             352378
at_plant_time          351018
u_Cicle                   209
name                     2063
Nombre del proyecto     23608
ship_addr_line          73551
map_page                  953
dtype: int64


In [107]:
remissions.shape[0]

355802

In [108]:
# print unique values per column



Based on the initial unique values for each column:
- There have been 777 orders in total
- There have been 149 trucks
- There have been 22844 different projects
- There have been 2019 different clients (apparently)
- There have been 70753 different addresses for delivery

#### Imputation Section

In [109]:
# 1) Parse dates (coerce invalid to NaT so we can measure failures)
remissions['order_date_parsed'] = pd.to_datetime(
    remissions['order_date'], errors='coerce'
)

# 2) % of rows that failed to parse
remissions['order_date_parsed'].isna().mean()

np.float64(0.0)

#### Exporting Cleaned Dataset

In [110]:
# Has to be xlsx because of datetime format
remissions.to_excel("../data/processed/remissions_db_cleaned.xlsx", index=False)